In [1]:
# Colab bootstrap: clones the repo and installs deps. Does nothing when run locally from the repo.
import os, sys
if 'google.colab' in sys.modules and not os.path.exists('ranker.py'):
    !git clone -q https://github.com/beckortikov/dsk-product-ranker
    %cd dsk-product-ranker
    %pip install -q -r requirements.txt
    print('ready')

# DSK Bank — Product Cards Ranker · demo & evaluation

[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/beckortikov/dsk-product-ranker/blob/main/demo.ipynb) · [code on GitHub](https://github.com/beckortikov/dsk-product-ranker) · [live dashboard on Hugging Face](https://beckortikov-dsk-product-ranker.hf.space/dashboard/)

Lightweight bilingual (BG + EN) ranker for the Smart Search on dskbank.bg.
Zero ML at runtime, sub-millisecond, catalog is a hot-reloadable `products.json`.

0. The raw data: `df_data.csv` + `df_mapping.csv` — what is in there and what it implies
1. Building the catalog from the raw data (offline step) — one card per product
2. Query → ranked cards (+ interactive widget)
3. How a query is understood (prefix / typo / transliteration)
4. Evaluation: ours vs. lexical ablations vs. multilingual embeddings; dev set and held-out
5. Latency
6. Add / delete a product at runtime

In [2]:
import json, sys, time, re
from pathlib import Path
import pandas as pd
sys.path.insert(0, str(Path('.').resolve()))
pd.set_option('display.max_colwidth', 90)

df = pd.read_csv('df_data.csv')          # scraped product pages
mp = pd.read_csv('df_mapping.csv')       # BG page -> EN page
print('df_data.csv   ', df.shape, list(df.columns))
print('df_mapping.csv', mp.shape, list(mp.columns))
df.head(3)[['document_id', 'document_lang', 'product_name', 'document_url']]

df_data.csv    (201, 5) ['document_id', 'document_url', 'document_lang', 'document_text', 'product_name']
df_mapping.csv (105, 5) ['bg_document_id', 'bg_url', 'en_url', 'match_status', 'en_document_id']


,document_id,document_lang,product_name,document_url
0,11299,bg,Стандартна разплащателна сметка,https://dskbank.bg/индивидуални-клиенти/разплащания/сметки/детайли-за-разплащателни-пр...
1,11407,bg,Стандартен потребителски кредит,https://dskbank.bg/индивидуални-клиенти/кредитиране/потребителски-кредити/детайли-за-п...
2,10217,bg,DSK Mobile,https://dskbank.bg/индивидуални-клиенти/електронно-банкиране/dsk-mobile


## 0. The raw data

Before any model: what is actually in the two files, because every design decision below comes from here.

In [3]:
print('pages by language:', df.document_lang.value_counts().to_dict())
print('mapping status:    ', mp.match_status.value_counts().to_dict())
L = df.document_text.str.len()
print(f'page length (chars): median {int(L.median())}, p10 {int(L.quantile(.1))}, p90 {int(L.quantile(.9))}  -> long landing pages, not cards')
print(f'<img alt> tags per page: median {int(df.document_text.str.count("<img").median())}      -> markup noise to clean')
print(f'unique product_name: {df.product_name.nunique()} of {len(df)} pages          -> near-duplicates (client-type variants)')

pages by language: {'bg': 105, 'en': 96}
mapping status:     {'MATCHED': 96, 'UNMATCHED': 9}
page length (chars): median 4096, p10 1902, p90 10000  -> long landing pages, not cards
<img alt> tags per page: median 12      -> markup noise to clean
unique product_name: 164 of 201 pages          -> near-duplicates (client-type variants)


**Bulgarian-only pages.** 9 pages have no English counterpart, and two of them are flagships from the spec (DSK Mobile, DSK Online). An English query must still find them, so BG and EN are indexed together in one card rather than as two indexes.

In [4]:
unmatched = mp[mp.match_status != 'MATCHED'].merge(df[['document_id', 'product_name']], left_on='bg_document_id', right_on='document_id')
unmatched[['bg_document_id', 'product_name']]

,bg_document_id,product_name
0,11307,Виртуална спестовна компонента
1,11706,Кредит за покупка и довършителни работи
2,11387,Твоят нов дом в България
3,10228,DSK Online
4,10217,DSK Mobile
5,10561,Кредит „Експерт“
6,10541,Кредитна карта Visa Business Gold
7,10538,Бизнес дебитни карти
8,10438,Виртуални карти за бизнес клиенти


**Near-duplicates.** The same product is published for individual / business / corporate clients, up to six pages per product:

In [5]:
dup = df.groupby('product_name').document_id.agg(list)
dup[dup.str.len() > 2].to_frame('document_ids')

,document_ids
product_name,
DSK Business,"[12336, 10572, 10836]"
DSK Direct,"[12293, 10835, 12301]"
DSK MC@Sign,"[16303, 16220, 16300, 16219]"
DSK mToken,"[11371, 10575, 12333, 12297, 10819, 12308]"
Multicash,"[10556, 12335, 10818, 12300]"
ДСК Директ,"[12331, 12363, 17075]"


**The selling message is in the data after all.** Every page has a `#` title followed by a `##` hero tagline — the bank's own one-line pitch, exactly the `product_summary` the spec asks for. Raw page for DSK mToken:

In [6]:
raw = df.loc[df.document_id == 10575, 'document_text'].iloc[0]
print('\n'.join(l for l in raw.split('\n') if l.strip())[:700])

<img alt="DSK mToken приложение">
# DSK mToken за бизнес клиенти
## Мобилно приложение за нареждане и потвърждаване на преводи
<img alt="">
Бизнес клиенти
Моят бизнес
Електронно банкиране
DSK mToken
<img alt="DSK mToken">
## Какво е DSK mToken?
DSK mToken е мобилно приложение за потвърждаване на преводи и документи, наредени през онлайн банкирането ДСК Директ за индивидуални и бизнес клиенти, както и за потвърждаване на плащания с банкови карти.
<img alt="">
<img alt="">
### Какви са Вашите предимства?
<img alt="">
Активирането на DSK mToken е безплатно и се инсталира на телефона Ви
<img alt="">
Висока степен на сигурност - двуфакторна защита на извършваната операция
<img alt="">
Потвърждава


## 1. Building the catalog (offline)

`build_catalog.py` turns the two CSVs into `products.json`: pairs BG↔EN via the mapping, collapses client-type variants into one card, strips markup and breadcrumbs, extracts the tagline, derives a category from the URL, adds curated aliases and TF-IDF keywords, and attaches the flagship config. Run it right here:

In [7]:
from build_catalog import build
t0 = time.perf_counter()
cards = build()
print(f'{len(df)} raw pages -> {len(cards)} product cards in {time.perf_counter()-t0:.1f} s')
print('cards merged from 3+ pages:', sum(1 for c in cards if len(c['document_ids']) > 2),
      '| BG-only cards:', sum(1 for c in cards if not c['product_name_en']),
      '| summary from hero tagline:', sum(1 for c in cards if c['summary_source'] == 'tagline'))
pd.DataFrame([{'card': c['canonical_key'], 'document_ids': c['document_ids'], 'summary_bg': c['summary_bg']}
              for c in cards if c['canonical_key'] in ('dsk mtoken', 'dsk mobile', 'физически пос терминал', 'кредитна карта galaxy')])

201 raw pages -> 85 product cards in 0.2 s
cards merged from 3+ pages: 18 | BG-only cards: 9 | summary from hero tagline: 81


,card,document_ids,summary_bg
0,dsk mobile,[10217],"Повече възможности, стабилност и лекота"
1,dsk mtoken,"[10575, 10819, 11371, 12297, 12308, 12333]",Подписваш преводи и документи по-лесно от всякога
2,кредитна карта galaxy,"[11028, 11684]",Без годишна такса при 24 трансакции през годината
3,физически пос терминал,"[10523, 10786, 12337, 12393]","необходимо да направя, за да имам физически ПОС от Банка ДСК?"


In [8]:
# The runtime never sees the CSVs: it only reads this file.
Path('products.json').write_text(json.dumps(cards, ensure_ascii=False, indent=2), encoding='utf-8')
from ranker import ProductCardRanker
t0 = time.perf_counter(); ranker = ProductCardRanker(cards)
print(f'{len(ranker)} cards indexed in {(time.perf_counter()-t0)*1000:.0f} ms')

85 cards indexed in 141 ms


### What a card looks like

One card per product. Client-type variants are collapsed into one card that keeps every `document_id`; the first one is returned as the representative.

In [9]:
c = next(c for c in cards if c['canonical_key'] == 'dsk mtoken')
{k: (v if not isinstance(v, str) or len(v) < 120 else v[:120] + '…') for k, v in c.items() if k not in ('text_bg', 'text_en')}

{'card_id': 'dsk_mtoken',
 'canonical_key': 'dsk mtoken',
 'document_ids': [10575, 10819, 11371, 12297, 12308, 12333],
 'product_name_bg': 'DSK mToken',
 'product_name_en': 'DSK mToken',
 'name_variants': ['DSK mToken'],
 'category': 'electronic mobile applications internet banking small and medium business electronic banking електронно банкиране електр…',
 'summary_bg': 'Подписваш преводи и документи по-лесно от всякога',
 'summary_en': 'With DSK mToken signing bank transfers anddocuments is easier than ever',
 'summary_source': 'tagline',
 'urls': ['https://dskbank.bg/en/business-clients/small-and-medium-business/electronic-banking/dsk-mtoken',
  'https://dskbank.bg/en/corporate-clients/corporate-clients/internet-banking/dsk-mtoken',
  'https://dskbank.bg/en/individual-clients/electronic/mobile-applications/dsk-mtoken',
  'https://dskbank.bg/бизнес-клиенти/корпоративни-клиенти/продукти-и-услуги/електронно-банкиране/dsk-mtoken',
  'https://dskbank.bg/бизнес-клиенти/моят-бизнес/електро

## 2. Query → ranked cards

Response contract (per item): `document_id`, `product_name`, `product_summary`, `relevance`.

In [10]:
def show(q, k=5):
    print(f'\n=== {q!r}')
    for r in ranker.rank(q, top_k=k):
        print(f"  {r['relevance']:.2f}  [{r['document_id']:>5}]  {r['product_name']:<45}  {r['product_summary'][:60]}")

for q in ['DSK Mobile', 'mobile', 'мобилно банкиране', 'DSK Smart', 'дск директ',
          'кредитна карта', 'ипотечен кредит', 'student loan', 'home insurance',
          'dsk mob',            # mid-typing
          'кредитна крата',     # typo
          'dsk mobail',         # transliteration-ish
          'такси за превод',    # weakly related → low relevance
          'зззз']:              # nothing
    show(q)


=== 'DSK Mobile'
  1.00  [10217]  DSK Mobile                                     Повече възможности, стабилност и лекота
  0.69  [10572]  DSK Business                                   Едно приложение - много решения за бизнеса Ви
  0.65  [10229]  DSK Smart                                      Банкиране на Банка ДСК за индивидуални клиенти
  0.61  [10575]  DSK mToken                                     Подписваш преводи и документи по-лесно от всякога
  0.50  [10228]  DSK Online                                     Дигитално банкиране през твоя лаптоп или компютър

=== 'mobile'
  1.00  [10217]  DSK Mobile                                     Повече възможности, стабилност и лекота
  0.59  [10572]  DSK Business                                   Едно приложение - много решения за бизнеса Ви
  0.52  [10229]  DSK Smart                                      Банкиране на Банка ДСК за индивидуални клиенти
  0.48  [10575]  DSK mToken                                     Подписваш преводи и докуме

In [11]:
try:
    import ipywidgets as widgets
    from IPython.display import display, clear_output
    box = widgets.Text(description='query:', placeholder='type as a user would…', layout=widgets.Layout(width='600px'))
    out = widgets.Output()
    def on_change(change):
        with out:
            clear_output()
            for r in ranker.rank(change['new'], top_k=5):
                print(f"{r['relevance']:.2f}  [{r['document_id']}]  {r['product_name']}  —  {r['product_summary'][:70]}")
    box.observe(on_change, names='value')
    display(box, out)
except Exception as e:
    print('widget unavailable:', e)

Text(value='', description='query:', layout=Layout(width='600px'), placeholder='type as a user would…')

Output()

## 3. How a query is understood

Each token is mapped to catalog terms: exact → prefix (last token only) → transliteration → fuzzy (Damerau-Levenshtein ≤ 1–2). Match quality discounts the contribution.

In [12]:
for q in ['dsk mob', 'кредитна крата', 'kreditna karta', 'депозит', 'mtokn']:
    print(f'{q!r:20}', [[(m.term, m.quality) for m in g][:5] for g in ranker.analyze(q)])

'dsk mob'            [[('dsk', 1.0)], [('mobil', 0.85), ('мобилно', 0.75), ('мобилн', 0.75), ('мобайл', 0.75)]]
'кредитна крата'     [[('кредитн', 1.0)], [('кражб', 0.85), ('края', 0.85), ('крайна', 0.85), ('краткотрайн', 0.85), ('кратк', 0.85)]]
'kreditna karta'     [[('кредитн', 0.75)], [('кар', 0.75)]]
'депозит'            [[('депозит', 1.0), ('депозитарна', 0.85), ('депозитн', 0.85), ('deposit', 0.6)]]
'mtokn'              [[('mtoken', 0.6)]]


## 4. Evaluation

No query logs exist, so `eval.py` holds a hand-curated bilingual **dev set** (73 queries tagged
*flagship / generic / product / prefix / xlang / typo / translit / long* + 10 out-of-catalog negatives).

* **Hit@1 / Hit@3 / MRR** on positives; **abstain** = share of negatives returned empty or below threshold.

Systems: BM25F only → + query fallbacks → **ours** (+ curated aliases, pins, boosts) → multilingual sentence embeddings as the *reference point* the spec allows (not part of the final ranker).

In [13]:
from eval import EVAL, NEGATIVES, evaluate, latency, lexical_ablation

rows = []
for name, r in [('BM25F only', lexical_ablation(cards, boosts=False, fuzzy=False)),
                ('BM25F + prefix/typo/translit', lexical_ablation(cards, boosts=False, fuzzy=True)),
                ('Ours', ProductCardRanker(cards))]:
    m = evaluate(lambda q, k: r.rank(q, k, min_relevance=0.0), cards)
    lat = latency(lambda q, k: r.rank(q, k), n=500)
    if hasattr(r, '_restore'): r._restore()
    rows.append({'system': name, 'Hit@1': m['hit@1'], 'Hit@3': m['hit@3'], 'MRR': m['mrr'], 'abstain': m['abstain'],
                 'p50 ms': lat['p50_ms'], 'p95 ms': lat['p95_ms'], **{f'{t}': v for t, v in m['by_tag'].items()}})
pd.DataFrame(rows).set_index('system').round(3)

,Hit@1,Hit@3,MRR,abstain,p50 ms,p95 ms,flagship,generic,prefix,product,xlang,typo,translit,long
system,,,,,,,,,,,,,,
BM25F only,0.822,0.877,0.857,0.8,0.082,0.123,0.714,0.667,1.0,1.0,0.947,0.4,0.2,1.0
BM25F + prefix/typo/translit,0.904,0.973,0.937,0.8,0.103,0.284,0.786,0.667,1.0,1.0,1.000,0.6,0.8,1.0
Ours,0.986,1.000,0.993,0.8,0.098,0.282,1.000,1.000,1.0,1.0,1.000,0.8,1.0,1.0


**Held-out check.** The 73 queries above were written by the same person who tuned the ranker. `eval.py --holdout` generates 770 queries mechanically from the catalog (full names, 55 % prefixes, typos, transliterations, 2-word subsets, taglines) and is never tuned against — this is the honest number.

In [14]:
from eval import run_holdout
_ = run_holdout(cards)


#### HELD-OUT set: 770 generated queries (never tuned against)
BM25F only                       Hit@1 76.5%  Hit@3 86.8%  MRR 0.816
   by tag: {'name_bg': '100%', 'name_en': '99%', 'prefix_bg': '74%', 'prefix_en': '80%', 'subset_bg': '80%', 'tagline_bg': '90%', 'tagline_en': '88%', 'translit_bg': '16%', 'typo_bg': '73%', 'typo_en': '71%'}


BM25F + prefix/typo/translit     Hit@1 91.7%  Hit@3 98.4%  MRR 0.950
   by tag: {'name_bg': '100%', 'name_en': '99%', 'prefix_bg': '78%', 'prefix_en': '86%', 'subset_bg': '80%', 'tagline_bg': '90%', 'tagline_en': '88%', 'translit_bg': '95%', 'typo_bg': '99%', 'typo_en': '99%'}
Ours                             Hit@1 91.9%  Hit@3 98.4%  MRR 0.952
   by tag: {'name_bg': '100%', 'name_en': '99%', 'prefix_bg': '78%', 'prefix_en': '86%', 'subset_bg': '80%', 'tagline_bg': '90%', 'tagline_en': '88%', 'translit_bg': '98%', 'typo_bg': '99%', 'typo_en': '99%'}


In [15]:
# Reference point: multilingual embeddings (downloads ~120 MB on first run; skip if offline)
try:
    from eval import EmbeddingRanker
    e = EmbeddingRanker(cards)
    m = evaluate(e.rank, cards, min_relevance=0.45)
    lat = latency(e.rank, n=100)
    print(f"Embeddings  Hit@1 {m['hit@1']:.1%}  Hit@3 {m['hit@3']:.1%}  MRR {m['mrr']:.3f}  p50 {lat['p50_ms']:.1f} ms")
    print('by tag:', {t: f'{v:.0%}' for t, v in m['by_tag'].items()})
except Exception as exc:
    print('embedding baseline skipped:', exc)

/Users/behzod/Documents/test/.venv/lib/python3.9/site-packages/urllib3/__init__.py:35: NotOpenSSLWarning: urllib3 v2 only supports OpenSSL 1.1.1+, currently the 'ssl' module is compiled with 'LibreSSL 2.8.3'. See: https://github.com/urllib3/urllib3/issues/3020
  warnings.warn(
/Users/behzod/Documents/test/eval.py:206: UserWarning: The model sentence-transformers/paraphrase-multilingual-MiniLM-L12-v2 now uses mean pooling instead of CLS embedding. In order to preserve the previous behaviour, consider either pinning fastembed version to 0.5.1 or using `add_custom_model` functionality.
  self.model = TextEmbedding(model_name=model)


/Users/behzod/Documents/test/eval.py:230: RuntimeWarning: divide by zero encountered in matmul
  sims = self.doc_emb @ q
/Users/behzod/Documents/test/eval.py:230: RuntimeWarning: overflow encountered in matmul
  sims = self.doc_emb @ q
/Users/behzod/Documents/test/eval.py:230: RuntimeWarning: invalid value encountered in matmul
  sims = self.doc_emb @ q


Embeddings  Hit@1 56.2%  Hit@3 74.0%  MRR 0.648  p50 23.0 ms
by tag: {'flagship': '29%', 'generic': '100%', 'prefix': '17%', 'product': '65%', 'xlang': '84%', 'typo': '40%', 'translit': '40%', 'long': '50%'}


## 5. Latency

In [16]:
import statistics
qs = [q for q, _, _ in EVAL] * 30
ts = []
for q in qs:
    t0 = time.perf_counter(); ranker.rank(q, 5); ts.append((time.perf_counter() - t0) * 1000)
ts.sort()
print(f'p50 {statistics.median(ts):.2f} ms | p95 {ts[int(.95*len(ts))]:.2f} ms | p99 {ts[int(.99*len(ts))]:.2f} ms  over {len(ts)} queries, {len(ranker)} cards')

p50 0.80 ms | p95 3.07 ms | p99 6.41 ms  over 2190 queries, 85 cards


## 6. Add / delete a product at runtime

A card is a dict. No retraining: the index rebuilds in-process in ~0.1 s. In production the same happens through `PUT/DELETE /catalog/cards/{id}` or by editing `products.json`.

In [17]:
new = {'card_id': 'зелена_ипотека', 'canonical_key': 'зелена ипотека', 'document_ids': [999999],
       'product_name_bg': 'Зелена ипотека', 'product_name_en': 'Green mortgage',
       'name_variants': ['Зелена ипотека', 'Green mortgage'], 'aliases': ['еко кредит', 'енергийно ефективен дом'],
       'summary_bg': 'По-ниска лихва за енергийно ефективен дом', 'summary_en': 'Lower rate for an energy-efficient home'}
t0 = time.perf_counter(); r2 = ProductCardRanker(cards + [new]); build_ms = (time.perf_counter() - t0) * 1000
print(f'rebuilt {len(r2)} cards in {build_ms:.0f} ms')
for q in ['зелена ипотека', 'green mortgage', 'еко кредит']:
    print(q, '→', r2.rank(q, 1)[0]['product_name'], r2.rank(q, 1)[0]['relevance'])

rebuilt 86 cards in 268 ms
зелена ипотека → Зелена ипотека 1.0
green mortgage → Зелена ипотека 1.0
еко кредит → Зелена ипотека 0.95
